[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/infuze.ipynb)

# Infuzní směs: táž úloha dvakrát

Notebook k bloku 3 druhého cvičení: úloha, která se právě spočítala na tabuli,
zapsaná dvěma způsoby — **symbolicky** v CVXPY a ve **standardním tvaru**, který
chce `scipy.optimize.linprog`. Maticové rozšíření téže úlohy na šest přípravků
je v navazujícím notebooku [`matice.ipynb`](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/matice.ipynb).

## Zadání slovy

> Míchá se infuzní směs ze dvou roztoků glukózy. Roztok A má 50 g glukózy
> v litru a stojí 20 Kč/l, roztok B má 200 g/l a stojí 50 Kč/l. Pacient má
> dostat **aspoň 150 g glukózy** a **aspoň 1,5 l objemu**.
>
> **Kolik litrů každého roztoku namíchat, aby to bylo co nejlevnější?**

## Formulace

$$
\begin{aligned}
\text{minimize}_{\mathbf x}\quad & 20x_1 + 50x_2 && \text{cena, Kc}\\
\text{subject to}\quad & 50x_1 + 200x_2 \ge 150 && \text{glukoza, g}\\
& x_1 + x_2 \ge 1{,}5 && \text{objem, l}\\
& \mathbf x \ge \mathbf 0 && \text{objemy nejsou zaporne}
\end{aligned}
$$

Na tabuli vyšlo $\mathbf x^\star = (1;\ 0{,}5)$ l za **45 Kč**, obě omezení jsou
v optimu aktivní a stínové ceny jsou **0,20 Kč za gram** glukózy a **10 Kč za
litr** objemu. Tahle čísla jsou dál měřítko: kód, který je nepotvrdí, je špatně.

## Jak se s notebookem pracuje

Doplňte **tři místa `# TODO`**: model v CVXPY, kontrolu dosazením
a převod omezení pro `linprog`. Volání solverů a graf jsou připravené.

| v zadání | v kódu |
|---|---|
| $\mathbf x \ge \mathbf 0$ | `x = cp.Variable(2, nonneg=True)` |
| $\mathbf c^\mathsf T\mathbf x$ | `CENA @ x` |
| $\min$ | `cp.Minimize(...)` |
| $50x_1 + 200x_2 \ge 150$ | `GLU @ x >= POZ_GLU` |
| $x_1 + x_2 \ge 1{,}5$ | `OBJ @ x >= POZ_OBJ` |
| celá úloha | `cp.Problem(ucel, omezeni)` |
| stínová cena omezení | `omezeni[0].dual_value` |

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/tabule_infuze.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/tabule_infuze.py). Notebook sám staví [`kod/postav_notebooky.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/postav_notebooky.py) — ruční úprava `.ipynb` se při dalším spuštění přepíše.*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linprog

## Data úlohy

Vektory obsahují cenu, glukózu a objem na litr každého roztoku.

In [ ]:
CENA = np.array([20.0, 50.0])     # Kč za litr roztoku A a B
GLU = np.array([50.0, 200.0])     # g glukózy v litru roztoku A a B
OBJ = np.array([1.0, 1.0])        # l objemu v litru roztoku (triviálně po jednom)
POZ_GLU = 150.0                   # požadavek na glukózu [g]
POZ_OBJ = 1.5                     # požadavek na objem [l]

### Krok 1 — formulace v CVXPY *(díra 1)*

In [ ]:
# TODO 1: přepište formulaci z tabule do CVXPY.
#   proměnná ... dva objemy v litrech, oba nezáporné
#   účelová  ... cena celé směsi, tedy skalární součin CENA a x
#   omezení  ... seznam DVOU nerovnic (glukóza, objem), obě typu >=
x = ...        # cp.Variable(...)
ucel = ...     # cp.Minimize(...)
omezeni = ...  # [..., ...]

In [ ]:
uloha = cp.Problem(ucel, omezeni)
# pevný solver pro reprodukovatelné výsledky
uloha.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_gap_rel=1e-11, tol_feas=1e-11)
x_cvxpy = np.asarray(x.value).ravel()
cena_opt = float(uloha.value)

print(f"stav řešení: {uloha.status}")           # 'optimal' -- jinak se dál nepočítá
print(f"x* = ({x_cvxpy[0]:.4f}; {x_cvxpy[1]:.4f}) l")
print(f"cena = {uloha.value:.4f} Kč   (na tabuli vyšlo 45 Kč)")

## Kontrola výsledku *(díra 2)*

Dosadíme výsledek do požadavků na glukózu a objem. Srovnáme ho také s ručním výpočtem.

In [ ]:
# TODO 2: spočítejte glukózu a objem z nalezených objemů x_cvxpy.
glukoza = ...
objem = ...
# Doplňte dvě podmínky: požadavek má být splněn s tolerancí 1e-6.
assert ..., "řešení nedodá dost glukózy"
assert ..., "řešení nedodá dost objemu"

In [ ]:
print(f"glukóza {glukoza:.6f} g (požadavek {POZ_GLU})")
print(f"objem   {objem:.6f} l (požadavek {POZ_OBJ})")
x_vrchol = np.linalg.solve(np.vstack([GLU, OBJ]), np.array([POZ_GLU, POZ_OBJ]))
print(f"průsečík obou omezení tužkou: ({x_vrchol[0]:.6f}; {x_vrchol[1]:.6f})")
assert np.allclose(x_cvxpy, x_vrchol, atol=1e-6), "optimum neleží v průsečíku omezení"

# kontrola umí spadnout: (3; 0) je taky přípustné, ale optimum to není
print(f"bod (3; 0): glukóza {GLU @ [3.0, 0.0]:.0f} g, objem {OBJ @ [3.0, 0.0]:.1f} l, "
      f"cena {CENA @ [3.0, 0.0]:.0f} Kč — přípustné, ale o 15 Kč dražší")

## Stínové ceny

Solver vrací kromě řešení ještě druhé číslo ke každému omezení: o kolik by se
zdražilo optimum, kdyby se požadavek přitvrdil o jednotku. Na tabuli vyšlo
0,20 Kč/g a 10 Kč/l — teď se to má potvrdit.

Duál se **nikdy nebere na slovo**: druhá půlka buňky ho ověří přepočtem, tedy
tím, že se úloha vyřeší znovu s požadavkem o gram vyšším.

In [ ]:
y_glu = float(omezeni[0].dual_value)
y_obj = float(omezeni[1].dual_value)
print(f"stínová cena glukózy: {y_glu:.6f} Kč/g   (ruční předpověď 0,20)")
print(f"stínová cena objemu:  {y_obj:.6f} Kč/l   (ruční předpověď 10)")
assert abs(y_glu - 0.20) < 1e-6, "duál u glukózy nesedí s ručním výpočtem"
assert abs(y_obj - 10.0) < 1e-6, "duál u objemu nesedí s ručním výpočtem"

uloha_tvrdsi = cp.Problem(cp.Minimize(CENA @ x),
                          [GLU @ x >= POZ_GLU + 1.0, OBJ @ x >= POZ_OBJ])
uloha_tvrdsi.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_gap_rel=1e-11, tol_feas=1e-11)
print(f"151 g místo 150 g: {uloha_tvrdsi.value:.4f} Kč, tedy o "
      f"{uloha_tvrdsi.value - cena_opt:.4f} Kč víc")
print("to je přesně stínová cena — a takhle se duál ověřuje, ne vírou")

## Převod pro `linprog` *(díra 3)*

`linprog` přijímá nerovnosti ve tvaru $A\mathbf x\le\mathbf b$.
Naše omezení mají opačný směr, proto **obě strany** vynásobíme −1.
Doplňte matici a pravou stranu; volání solveru níže je hotové.

In [ ]:
# TODO 3: převeďte obě nerovnosti >= na tvar A_ub @ x <= b_ub.
# Řádky spojíte pomocí np.vstack([GLU, OBJ]); změňte i pravou stranu.
A_ub = ...
b_ub = ...

In [ ]:
res = linprog(c=CENA, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)])
x_scipy = res.x
print(f"status: {res.message.strip()}")
print(f"x* = ({x_scipy[0]:.4f}; {x_scipy[1]:.4f}) l, cena = {res.fun:.4f} Kč")
# marginály patří k b_ub, a b_ub jsme otočili -- proto se otočí zpátky mínusem
print(f"stínové ceny: {-res.ineqlin.marginals[0]:.4f} Kč/g, "
      f"{-res.ineqlin.marginals[1]:.4f} Kč/l")

## Porovnání výsledků

Obě implementace mají vrátit stejný výsledek. Jejich shoda ale neodhalí chybu, kterou jsme zapsali do obou modelů.

In [ ]:
print(f"CVXPY  : x1 = {x_cvxpy[0]:.9f}  x2 = {x_cvxpy[1]:.9f}")
print(f"linprog: x1 = {x_scipy[0]:.9f}  x2 = {x_scipy[1]:.9f}")
print(f"největší rozdíl ve složkách: {np.max(np.abs(x_cvxpy - x_scipy)):.2e}")
assert np.allclose(x_cvxpy, x_scipy, atol=1e-9), "dvě cesty k témuž optimu se rozešly"
print("obě cesty vedou do téhož vrcholu — lišil se jen zápis, ne úloha")

## Obrázek: přípustná oblast, vrstevnice ceny a optimum

Dvě proměnné jsou poslední počet, u kterého se úloha dá nakreslit. Vrstevnice
ceny je přímka; posouvá se k počátku, dokud se přípustné oblasti jen dotýká —
a dotkne se jí ve vrcholu.

In [ ]:
xs = np.linspace(0.0, 3.4, 200)
h_glu = (POZ_GLU - GLU[0] * xs) / GLU[1]      # hranice omezení na glukózu
h_obj = (POZ_OBJ - OBJ[0] * xs) / OBJ[1]      # hranice omezení na objem

plt.figure(figsize=(7.4, 5.2))
plt.fill_between(xs, np.maximum(np.maximum(h_glu, h_obj), 0.0), 1.75,
                 color="#dbeafe", label="přípustná oblast")
plt.plot(xs, h_glu, color="#b45309", lw=2.2, label="glukóza: $50x_1+200x_2=150$")
plt.plot(xs, h_obj, color="#7c3aed", lw=2.2, label="objem: $x_1+x_2=1{,}5$")
for c in (30.0, 60.0):
    plt.plot(xs, (c - CENA[0] * xs) / CENA[1], color="#6b7280", ls=":", lw=1.1)
plt.plot(xs, (cena_opt - CENA[0] * xs) / CENA[1], color="#1a7f37", ls="--", lw=1.9,
         label=f"vrstevnice ceny {cena_opt:.0f} Kč")
plt.plot(x_cvxpy[0], x_cvxpy[1], "o", ms=13, mfc="none", mec="#1a7f37", mew=2.6,
         label=f"optimum ({x_cvxpy[0]:.2f}; {x_cvxpy[1]:.2f})")
plt.xlim(0, 3.4)
plt.ylim(0, 1.75)
plt.xlabel("roztok A (50 g/l, 20 Kč/l)  $x_1$ [l]")
plt.ylabel("roztok B (200 g/l, 50 Kč/l)  $x_2$ [l]")
plt.title("Optimum je roh přípustné oblasti")
plt.grid(alpha=0.28)
plt.legend(loc="upper right", fontsize=9.5)
plt.show()

## Shrnutí

Model musí odpovídat zadání. Výsledek ověřujeme dosazením; stínovou cenu můžeme ověřit přepočtem s mírně změněným požadavkem.